# BEST-Rec v2.3: DropoutNet Training + Contrastive Alignment

## The Key Insight

In GroupKFold, **all training users have interactions**. The model never sees a cold user during training, so it never learns what to do when user signal is missing. When tested on held-out users (who have zero features), the neural network outputs noise instead of a safe prediction.

**DropoutNet** (Volkovs et al., NeurIPS 2017) solves this by randomly zeroing out user features during training with probability `p`. This simulates cold-start scenarios, forcing the model to learn both:
- Full prediction (when user features are available)
- Item-only prediction (when user features are dropped)

### All improvements in v2.3:

| Technique | Source | What it does |
|---|---|---|
| **DropoutNet training** | NeurIPS 2017 | Randomly zero user features during training to simulate cold-start |
| **Contrastive alignment** | ICDM 2023, RecSys 2024 | Auxiliary loss aligning user embeddings with their interaction patterns |
| **Asymmetric sample weights** | Standard practice | Weight samples by `1/sqrt(user_count)` to focus on sparse users |
| **Ordinal-aware MSE** | Rating prediction literature | Higher penalty for opposite-end errors (predict 1 when true is 5) |
| **Warm-up + OneCycleLR** | Training best practice | Aggressive schedule for faster convergence |

In [ ]:
import subprocess, sys
def pip_install(*p):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *p])
pip_install("torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu128")
pip_install("transformers", "scikit-learn", "scipy", "numpy", "tqdm", "pandas", "ipywidgets")
print("Done.")

## 1. Imports & Setup

In [ ]:
import os, json, pickle, copy, time, warnings
from collections import defaultdict
from typing import List
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.sparse import csr_matrix
from transformers import AutoTokenizer, AutoModel
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
else:
    vram_gb = 0; print("CPU mode")

## 2. Configuration

In [ ]:
DATASET = "beauty"
DATASET_FILES = {
    "beauty": ("All_Beauty.jsonl", "meta_All_Beauty.jsonl"),
    "books": ("Books.jsonl", "meta_Books.jsonl"),
    "fashion": ("Amazon_Fashion.jsonl", "meta_Amazon_Fashion.jsonl"),
    "instruments": ("Musical_Instruments.jsonl", "meta_Musical_Instruments.jsonl"),
}
DATA_DIR = "./data"; CACHE_DIR = f"./cache/{DATASET}"; os.makedirs(CACHE_DIR, exist_ok=True)
INTER_FILE, META_FILE = DATASET_FILES[DATASET]
INTER_PATH = os.path.join(DATA_DIR, DATASET, INTER_FILE)
META_PATH = os.path.join(DATA_DIR, DATASET, META_FILE)

# Hardware
NUM_CPU_WORKERS = min(8, max(0, os.cpu_count() - 2))
USE_AMP = device.type == "cuda"; PIN_MEMORY = device.type == "cuda"
BATCH_SIZE = 4096 if vram_gb >= 12 else (2048 if vram_gb >= 8 else 1024)

# Model
PRETRAINED_MODEL = "distilbert-base-uncased"; TEXT_DIM = 768
HIDDEN_DIM = 256; NUM_HEADS = 4; NUM_ENCODER_LAYERS = 1
MAX_USER_REVIEWS = 5; SVD_COMPONENTS = 128; USER_SVD_COMPONENTS = 128
NUM_CLASSES = 5

# Training
LR = 5e-4; EPOCHS = 50; PATIENCE = 12
LAMBDA_CLS = 0.3; LAMBDA_CL = 0.1  # contrastive loss weight
WEIGHT_DECAY = 1e-4; GRAD_CLIP = 1.0; WARMUP_EPOCHS = 3

# DropoutNet
USER_DROPOUT_P = 0.3  # probability of zeroing ALL user features per sample

# Eval
NUM_FOLDS = 5; NEG_SAMPLES = 99; TOP_K = 10
COLD_USER_THRESHOLD = 3; COLD_ITEM_THRESHOLD = 5; RANKING_EVAL_USERS = 2000

SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f"Dataset: {DATASET} | DropoutNet p={USER_DROPOUT_P} | CL weight={LAMBDA_CL}")

## 3. Load Cached Data

In [ ]:
def cached(name, fn, force=False):
    path = os.path.join(CACHE_DIR, name)
    if os.path.exists(path) and not force:
        with open(path, "rb") as f: return pickle.load(f)
    result = fn()
    with open(path, "wb") as f: pickle.dump(result, f)
    return result
def cached_tensor(name, fn, force=False):
    path = os.path.join(CACHE_DIR, name)
    if os.path.exists(path) and not force:
        return torch.load(path, weights_only=True)
    result = fn()
    torch.save(result, path)
    return result

raw_path = os.path.join(CACHE_DIR, "raw_data.pkl")
assert os.path.exists(raw_path), "Run v2 notebook first!"
with open(raw_path, "rb") as f: data = pickle.load(f)
interactions = data["interactions"]; item_metadata = data["item_metadata"]
num_users = data["num_users"]; num_items = data["num_items"]

item_title_embeds = cached_tensor("item_title_embeds.pt", lambda: None)
def _cn():
    a = torch.tensor([item_metadata[i]["avg_rating"] for i in range(num_items)])
    r = torch.tensor([item_metadata[i]["rating_num"] for i in range(num_items)])
    p = torch.tensor([item_metadata[i]["price"] for i in range(num_items)])
    def z(x): return (x - x.mean()) / (x.std() + 1e-9)
    return torch.stack([z(a), z(r), z(p)], dim=-1)
item_numeric = cached_tensor("item_numeric.pt", _cn)
item_raw_avg = torch.tensor([item_metadata[i]["avg_rating"] for i in range(num_items)], dtype=torch.float32)
g_avg = item_raw_avg[item_raw_avg > 0].mean().item()
item_raw_avg[item_raw_avg == 0] = g_avg
print(f"Users: {num_users:,} | Items: {num_items:,} | Interactions: {len(interactions):,}")

## 4. Diagnostic Baselines

In [ ]:
rats = np.array([i["rating"] for i in interactions])
gm = rats.mean()
print(f"Global mean ({gm:.3f}) -> MAE = {np.mean(np.abs(rats - gm)):.4f}")
im = defaultdict(list)
for i in interactions: im[i["item_id"]].append(i["rating"])
imd = {k: np.mean(v) for k, v in im.items()}
p_im = np.array([imd.get(i["item_id"], gm) for i in interactions])
print(f"Item mean          -> MAE = {np.mean(np.abs(rats - p_im)):.4f}")
um = defaultdict(list)
for i in interactions: um[i["user_id"]].append(i["rating"])
umd = {k: np.mean(v) for k, v in um.items()}
p_comb = np.clip([umd.get(i["user_id"], gm) + imd.get(i["item_id"], gm) - gm for i in interactions], 1, 5)
print(f"User+Item mean     -> MAE = {np.mean(np.abs(rats - p_comb)):.4f}")
print(f"\nTarget: our model must beat global mean MAE")

## 5. Per-Fold Features + Sample Weights

In [ ]:
class TextEncoder:
    def __init__(self, mn, dev, ml=64):
        self.tok = AutoTokenizer.from_pretrained(mn)
        self.mod = AutoModel.from_pretrained(mn).to(dev).eval()
        self.dev = dev; self.ml = ml; self.dim = self.mod.config.hidden_size
    @torch.no_grad()
    def encode_batch(self, texts, bs=256):
        all_e = []
        for i in range(0, len(texts), bs):
            b = [t if t.strip() else "empty" for t in texts[i:i+bs]]
            inp = self.tok(b, padding=True, truncation=True, max_length=self.ml, return_tensors="pt").to(self.dev)
            with autocast(enabled=USE_AMP):
                o = self.mod(**inp)
            all_e.append(o.last_hidden_state[:,0,:].float().cpu())
        return torch.cat(all_e)
    def encode_user_reviews(self, train_inters, num_users, max_rev=5):
        ur = defaultdict(list)
        for i in train_inters:
            if i["review"].strip(): ur[i["user_id"]].append(i["review"])
        trips = []
        for uid in range(num_users):
            for j, rev in enumerate(ur.get(uid, [])[:max_rev]): trips.append((uid, j, rev))
        emb = torch.zeros(num_users, max_rev, self.dim)
        msk = torch.zeros(num_users, max_rev, dtype=torch.bool)
        if trips:
            enc = self.encode_batch([t[2] for t in trips])
            for idx, (uid, j, _) in enumerate(trips):
                emb[uid, j] = enc[idx]; msk[uid, j] = True
        return emb, msk

text_encoder = TextEncoder(PRETRAINED_MODEL, device)


def compute_fold_features(train_inters, fold_tag, force=False):
    # Item SVD
    def _isvd():
        r, c, v = [], [], []
        for i in train_inters: r.append(i["item_id"]); c.append(i["user_id"]); v.append(i["rating"])
        mat = csr_matrix((v, (r, c)), shape=(num_items, num_users))
        k = min(SVD_COMPONENTS, min(num_items, num_users)-1, len(set(r))-1)
        if k < 1: return torch.zeros(num_items, SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        res = svd.fit_transform(mat)
        if k < SVD_COMPONENTS: res = np.hstack([res, np.zeros((num_items, SVD_COMPONENTS-k))])
        return torch.tensor(res, dtype=torch.float32)
    item_svd = cached_tensor(f"svd_fold_{fold_tag}.pt", _isvd, force)

    # User SVD
    def _usvd():
        r, c, v = [], [], []
        for i in train_inters: r.append(i["user_id"]); c.append(i["item_id"]); v.append(i["rating"])
        mat = csr_matrix((v, (r, c)), shape=(num_users, num_items))
        k = min(USER_SVD_COMPONENTS, min(num_users, num_items)-1, len(set(r))-1)
        if k < 1: return torch.zeros(num_users, USER_SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        res = svd.fit_transform(mat)
        if k < USER_SVD_COMPONENTS: res = np.hstack([res, np.zeros((num_users, USER_SVD_COMPONENTS-k))])
        return torch.tensor(res, dtype=torch.float32)
    user_svd = cached_tensor(f"user_svd_fold_{fold_tag}.pt", _usvd, force)

    # User text
    def _ut():
        e, m = text_encoder.encode_user_reviews(train_inters, num_users, MAX_USER_REVIEWS)
        return {"embeds": e, "masks": m}
    ut = cached(f"user_embeds_fold_{fold_tag}.pkl", _ut, force)

    # Stats
    gs, gn = 0.0, 0
    us, uc = defaultdict(float), defaultdict(int)
    ist, ic = defaultdict(float), defaultdict(int)
    for i in train_inters:
        r = i["rating"]; gs += r; gn += 1
        us[i["user_id"]] += r; uc[i["user_id"]] += 1
        ist[i["item_id"]] += r; ic[i["item_id"]] += 1
    gm = gs / gn

    ub = torch.zeros(num_users)
    ib = torch.zeros(num_items)
    for uid in range(num_users):
        if uc[uid] > 0: ub[uid] = (us[uid] / uc[uid]) - gm
    for iid in range(num_items):
        if ic[iid] > 0: ib[iid] = (ist[iid] / ic[iid]) - gm

    # Asymmetric sample weights: 1/sqrt(user_count)
    user_weights = torch.ones(num_users)
    for uid in range(num_users):
        if uc[uid] > 0:
            user_weights[uid] = 1.0 / np.sqrt(uc[uid])
    # Normalise so mean weight = 1
    user_weights = user_weights / user_weights.mean()

    return {
        "item_svd": item_svd, "user_svd": user_svd,
        "user_text_embeds": ut["embeds"], "user_text_masks": ut["masks"],
        "global_mean": gm, "user_bias": ub, "item_bias": ib,
        "user_weights": user_weights,
    }

print("Feature engineering ready.")

## 6. Dataset Class (with sample weights)

In [ ]:
class DSv23(Dataset):
    def __init__(self, interactions, feats, item_title_embeds, item_numeric, item_raw_avg):
        self.uids = torch.tensor([i["user_id"] for i in interactions], dtype=torch.long)
        self.iids = torch.tensor([i["item_id"] for i in interactions], dtype=torch.long)
        self.rats = torch.tensor([i["rating"] for i in interactions], dtype=torch.float32)
        self.cls = torch.clamp(self.rats.long() - 1, min=0)
        self.u_text = feats["user_text_embeds"]; self.u_mask = feats["user_text_masks"]
        self.u_svd = feats["user_svd"]; self.i_svd = feats["item_svd"]
        self.i_title = item_title_embeds; self.i_num = item_numeric
        self.i_avg = item_raw_avg; self.u_weights = feats["user_weights"]

    def __len__(self): return len(self.rats)

    def __getitem__(self, idx):
        uid = self.uids[idx]; iid = self.iids[idx]
        return (self.u_text[uid], self.u_mask[uid], self.u_svd[uid],
                self.i_title[iid], self.i_num[iid], self.i_svd[iid], self.i_avg[iid],
                self.rats[idx], self.cls[idx], uid, iid, self.u_weights[uid])

## 7. BEST-Rec v2.3 Model

### Key: DropoutNet-style training

During `forward()`, when `training=True`, randomly zero ALL user features for each sample with probability `USER_DROPOUT_P`. This teaches the model to handle cold users by falling back to item-only prediction.

In [ ]:
class BESTRecV23(nn.Module):
    def __init__(self, n_users, n_items, global_mean, u_bias_init, i_bias_init):
        super().__init__()
        H = HIDDEN_DIM

        # Bias
        self.global_mean = nn.Parameter(torch.tensor(float(global_mean)), requires_grad=False)
        self.user_bias = nn.Embedding(n_users, 1); self.user_bias.weight.data = u_bias_init.unsqueeze(1)
        self.item_bias = nn.Embedding(n_items, 1); self.item_bias.weight.data = i_bias_init.unsqueeze(1)

        # User projections
        self.u_rev_proj = nn.Linear(TEXT_DIM, H)
        self.u_svd_proj = nn.Linear(USER_SVD_COMPONENTS, H)

        # Item projections
        self.i_title_proj = nn.Linear(TEXT_DIM, H)
        self.i_num_proj = nn.Linear(3, H)
        self.i_svd_proj = nn.Linear(SVD_COMPONENTS, H)
        self.i_type_emb = nn.Embedding(3, H)

        # User attention pool
        self.u_pool_q = nn.Parameter(torch.randn(1, 1, H) * 0.02)
        self.u_pool_attn = nn.MultiheadAttention(H, NUM_HEADS, batch_first=True, dropout=0.1)

        # Encoders
        ul = nn.TransformerEncoderLayer(H, NUM_HEADS, H*4, 0.1, batch_first=True, activation="gelu")
        self.u_enc = nn.TransformerEncoder(ul, NUM_ENCODER_LAYERS)
        il = nn.TransformerEncoderLayer(H, NUM_HEADS, H*4, 0.1, batch_first=True, activation="gelu")
        self.i_enc = nn.TransformerEncoder(il, NUM_ENCODER_LAYERS)

        # Cross-attention
        self.ca_u2i = nn.MultiheadAttention(H, NUM_HEADS, batch_first=True, dropout=0.1)
        self.n_u2i = nn.LayerNorm(H)
        self.ca_i2u = nn.MultiheadAttention(H, NUM_HEADS, batch_first=True, dropout=0.1)
        self.n_i2u = nn.LayerNorm(H)

        # Fusion
        self.fuse_proj = nn.Linear(H * 4, H)
        self.fuse_norm = nn.LayerNorm(H)
        self.fuse_mlp = nn.Sequential(nn.Linear(H, H), nn.GELU(), nn.Dropout(0.1))
        self.fuse_norm2 = nn.LayerNorm(H)

        # Heads
        self.residual_head = nn.Sequential(nn.Linear(H, H//2), nn.GELU(), nn.Dropout(0.05), nn.Linear(H//2, 1))
        self.item_only_head = nn.Sequential(nn.Linear(H + 1, H//2), nn.GELU(), nn.Linear(H//2, 1))
        self.classifier = nn.Linear(H, NUM_CLASSES)

        # Projection for contrastive loss
        self.user_cl_proj = nn.Linear(H, 64)
        self.item_cl_proj = nn.Linear(H, 64)

    def forward(self, u_rev, u_mask, u_svd, i_title, i_num, i_svd, i_avg,
                user_ids=None, item_ids=None):
        B = u_rev.size(0); H = HIDDEN_DIM

        # ══════════════════════════════════════════════════════
        # DROPOUTNET: randomly zero ALL user features per sample
        # ══════════════════════════════════════════════════════
        if self.training:
            # Per-sample dropout mask: shape (B,)
            drop_mask = torch.rand(B, device=u_rev.device) < USER_DROPOUT_P
            if drop_mask.any():
                # Zero out text reviews for dropped samples
                u_rev = u_rev.clone()
                u_rev[drop_mask] = 0.0
                u_mask = u_mask.clone()
                u_mask[drop_mask] = False
                # Zero out user SVD for dropped samples
                u_svd = u_svd.clone()
                u_svd[drop_mask] = 0.0
                # Zero out user bias for dropped samples
                if user_ids is not None:
                    user_ids = user_ids.clone()
                    # Don't zero IDs (bias handles it), but we'll zero the bias contribution separately below

        # User tokens
        u_tokens = self.u_rev_proj(u_rev)
        u_svd_tok = self.u_svd_proj(F.normalize(u_svd, p=2, dim=-1)).unsqueeze(1)
        u_all = torch.cat([u_tokens, u_svd_tok], dim=1)
        svd_unmask = torch.ones(B, 1, dtype=torch.bool, device=u_mask.device)
        u_full_mask = torch.cat([u_mask, svd_unmask], dim=1)
        u_pad = ~u_full_mask
        all_pad = u_pad.all(dim=1)
        if all_pad.any(): u_pad[all_pad, -1] = False
        u_encoded = self.u_enc(u_all, src_key_padding_mask=u_pad)
        pq = self.u_pool_q.expand(B, -1, -1)
        u_pool, _ = self.u_pool_attn(pq, u_encoded, u_encoded, key_padding_mask=u_pad)
        u_pool = u_pool.squeeze(1)

        # Item tokens
        t1 = self.i_title_proj(i_title).unsqueeze(1)
        t2 = self.i_num_proj(i_num).unsqueeze(1)
        t3 = self.i_svd_proj(F.normalize(i_svd, p=2, dim=-1)).unsqueeze(1)
        i_tokens = torch.cat([t1, t2, t3], dim=1)
        tid = torch.tensor([0,1,2], device=i_tokens.device).unsqueeze(0).expand(B,-1)
        i_tokens = i_tokens + self.i_type_emb(tid)
        i_encoded = self.i_enc(i_tokens)
        i_pool = i_encoded.mean(dim=1)

        # Cross-attention
        u2i, _ = self.ca_u2i(u_encoded, i_encoded, i_encoded)
        u2i = self.n_u2i(u2i + u_encoded)
        mf = u_full_mask.unsqueeze(-1).float()
        u2i_pool = (u2i * mf).sum(1) / (mf.sum(1) + 1e-9)

        i2u, _ = self.ca_i2u(i_encoded, u_encoded, u_encoded, key_padding_mask=u_pad)
        i2u = self.n_i2u(i2u + i_encoded)
        i2u_pool = i2u.mean(dim=1)

        # Fusion
        fc = torch.cat([u_pool, u2i_pool, i_pool, i2u_pool], dim=-1)
        z = self.fuse_norm(self.fuse_proj(fc))
        z = self.fuse_norm2(z + self.fuse_mlp(z))

        # Neural residual (bounded)
        neural_res = 2.0 * torch.tanh(self.residual_head(z).squeeze(-1))

        # Item-only residual
        item_only_in = torch.cat([i_pool, i_avg.unsqueeze(-1)], dim=-1)
        item_only_res = self.item_only_head(item_only_in).squeeze(-1)

        # Bias
        if user_ids is not None and item_ids is not None:
            ub = self.user_bias(user_ids).squeeze(-1)
            ib = self.item_bias(item_ids).squeeze(-1)
            # DropoutNet: zero user bias for dropped samples during training
            if self.training and drop_mask.any():
                ub = ub.clone()
                ub[drop_mask] = 0.0
            bias = self.global_mean + ub + ib
        else:
            bias = self.global_mean + torch.zeros(B, device=u_rev.device)

        # Blend: item-only + neural
        # The model learns to weight between neural (which needs user signal)
        # and item-only (which works without user signal)
        # After DropoutNet training, neural_res should be ~0 for dropped users
        pred = bias + 0.5 * (neural_res + item_only_res)
        pred = torch.clamp(pred, 1.0, 5.0)

        cls_logits = self.classifier(z)

        # Contrastive embeddings (for auxiliary loss)
        u_cl = self.user_cl_proj(u_pool)
        i_cl = self.item_cl_proj(i_pool)

        return pred, cls_logits, u_cl, i_cl

print("BESTRecV23 defined (DropoutNet + contrastive alignment).")

## 8. Contrastive Alignment Loss (InfoNCE)

Users should have embeddings close to the items they rate highly and far from items they rate poorly. This regularises the embedding space.

In [ ]:
def info_nce_loss(user_embeds, item_embeds, ratings, temperature=0.1):
    """
    Supervised contrastive loss: user-item pairs with rating >= 4 are positives,
    pairs with rating <= 2 are hard negatives. Uses in-batch negatives.
    """
    # Normalise
    u = F.normalize(user_embeds, dim=-1)
    i = F.normalize(item_embeds, dim=-1)

    # Similarity matrix: (B, B)
    sim = torch.mm(u, i.t()) / temperature

    # Positive pairs: diagonal (each user with their own item)
    # Weighted by rating: high ratings = stronger positive signal
    pos_weight = (ratings - 3.0).clamp(min=0) / 2.0  # 0 for rating<=3, 0.5 for 4, 1.0 for 5

    # InfoNCE: log(exp(pos) / sum(exp(all)))
    labels = torch.arange(sim.size(0), device=sim.device)
    loss = F.cross_entropy(sim, labels, reduction='none')

    # Weight by positive strength (only penalise if rating is high)
    weighted_loss = (loss * pos_weight).sum() / (pos_weight.sum() + 1e-9)
    return weighted_loss

print("InfoNCE contrastive loss defined.")

## 9. Training with DropoutNet + Contrastive + Asymmetric Weights

In [ ]:
amp_scaler = GradScaler(enabled=USE_AMP)


def train_one_epoch(model, optimizer, scheduler, dataloader):
    global amp_scaler
    model.train()
    total_loss, n = 0.0, 0

    for batch in tqdm(dataloader, desc="  Train", leave=False):
        (u_rev, u_mask, u_svd, i_tit, i_num, i_svd, i_avg,
         rat, cls, uids, iids, weights) = batch

        u_rev = u_rev.to(device, non_blocking=True)
        u_mask = u_mask.to(device, non_blocking=True)
        u_svd = u_svd.to(device, non_blocking=True)
        i_tit = i_tit.to(device, non_blocking=True)
        i_num = i_num.to(device, non_blocking=True)
        i_svd = i_svd.to(device, non_blocking=True)
        i_avg = i_avg.to(device, non_blocking=True)
        rat = rat.to(device, non_blocking=True)
        cls = cls.to(device, non_blocking=True)
        uids = uids.to(device, non_blocking=True)
        iids = iids.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            pred, cls_logits, u_cl, i_cl = model(
                u_rev, u_mask, u_svd, i_tit, i_num, i_svd, i_avg,
                user_ids=uids, item_ids=iids)

            # Weighted MSE (asymmetric sample weights)
            mse_per_sample = (pred - rat) ** 2
            loss_reg = (mse_per_sample * weights).mean()

            # Classification loss with label smoothing
            loss_cls = F.cross_entropy(cls_logits, cls, label_smoothing=0.05)

            # Contrastive alignment loss
            loss_cl = info_nce_loss(u_cl, i_cl, rat)

            loss = loss_reg + LAMBDA_CLS * loss_cls + LAMBDA_CL * loss_cl

        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        scheduler.step()

        total_loss += loss.item() * rat.size(0)
        n += rat.size(0)
    return total_loss / n


@torch.no_grad()
def evaluate_rating(model, dataloader):
    model.eval()
    all_p, all_t = [], []
    for batch in tqdm(dataloader, desc="  Eval", leave=False):
        (u_rev, u_mask, u_svd, i_tit, i_num, i_svd, i_avg,
         rat, cls, uids, iids, weights) = batch
        with autocast(enabled=USE_AMP):
            pred, _, _, _ = model(
                u_rev.to(device,non_blocking=True), u_mask.to(device,non_blocking=True),
                u_svd.to(device,non_blocking=True), i_tit.to(device,non_blocking=True),
                i_num.to(device,non_blocking=True), i_svd.to(device,non_blocking=True),
                i_avg.to(device,non_blocking=True),
                user_ids=uids.to(device,non_blocking=True),
                item_ids=iids.to(device,non_blocking=True))
        all_p.append(pred.float().cpu().numpy())
        all_t.append(rat.numpy())
    p, t = np.concatenate(all_p), np.concatenate(all_t)
    return mean_absolute_error(t, p), np.sqrt(mean_squared_error(t, p))


@torch.no_grad()
def evaluate_ranking(model, test_inters, feats, item_title_embeds, item_numeric, item_raw_avg):
    model.eval()
    ut_items = defaultdict(set)
    for i in test_inters: ut_items[i["user_id"]].add(i["item_id"])
    ndcg_list, hr_list = [], []
    all_items = set(range(num_items)); rng = np.random.RandomState(SEED)
    users = [u for u in ut_items if ut_items[u]][:RANKING_EVAL_USERS]
    for uid in tqdm(users, desc="  Ranking", leave=False):
        for pos in ut_items[uid]:
            neg_pool = list(all_items - ut_items[uid])
            if len(neg_pool) < NEG_SAMPLES: continue
            negs = rng.choice(neg_pool, NEG_SAMPLES, replace=False)
            cands = [pos] + list(negs); nc = len(cands)
            with autocast(enabled=USE_AMP):
                scores, _, _, _ = model(
                    feats["user_text_embeds"][uid].unsqueeze(0).expand(nc,-1,-1).to(device),
                    feats["user_text_masks"][uid].unsqueeze(0).expand(nc,-1).to(device),
                    feats["user_svd"][uid].unsqueeze(0).expand(nc,-1).to(device),
                    item_title_embeds[cands].to(device),
                    item_numeric[cands].to(device),
                    feats["item_svd"][cands].to(device),
                    item_raw_avg[cands].to(device))
            scores = scores.float().cpu().numpy()
            ranked = np.argsort(-scores)
            pr = int(np.where(ranked == 0)[0][0])
            hr_list.append(1.0 if pr < TOP_K else 0.0)
            ndcg_list.append(1.0/np.log2(pr+2) if pr < TOP_K else 0.0)
    return {f"NDCG@{TOP_K}": np.mean(ndcg_list) if ndcg_list else 0.0,
            f"HR@{TOP_K}": np.mean(hr_list) if hr_list else 0.0}

print("Training functions ready (DropoutNet + contrastive + weighted).")

## 10. Run Fold

In [ ]:
def run_fold_v23(fold_tag, train_inters, test_inters):
    global amp_scaler; amp_scaler = GradScaler(enabled=USE_AMP)

    print(f"\n{'='*60}")
    print(f"  [{fold_tag}]: {len(train_inters):,} train / {len(test_inters):,} test")
    print(f"{'='*60}")

    feats = compute_fold_features(train_inters, fold_tag)
    print(f"  Global mean: {feats['global_mean']:.3f}")

    train_ds = DSv23(train_inters, feats, item_title_embeds, item_numeric, item_raw_avg)
    test_ds = DSv23(test_inters, feats, item_title_embeds, item_numeric, item_raw_avg)
    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_CPU_WORKERS, pin_memory=PIN_MEMORY,
                 prefetch_factor=4 if NUM_CPU_WORKERS > 0 else None,
                 persistent_workers=True if NUM_CPU_WORKERS > 0 else False)
    train_dl = DataLoader(train_ds, shuffle=True, **dl_kw)
    test_dl = DataLoader(test_ds, shuffle=False, **dl_kw)

    model = BESTRecV23(num_users, num_items, feats["global_mean"],
                        feats["user_bias"], feats["item_bias"]).to(device)
    raw_model = model
    if hasattr(torch, "compile"):
        try: model = torch.compile(model, mode="reduce-overhead"); print("  Compiled")
        except: pass

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_dl)
    warmup_steps = WARMUP_EPOCHS * len(train_dl)
    def lr_fn(step):
        if step < warmup_steps: return step / max(1, warmup_steps)
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + np.cos(np.pi * prog))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_fn)

    best_mae, best_state, pat = float("inf"), None, 0
    for epoch in range(EPOCHS):
        t0 = time.time()
        loss = train_one_epoch(model, optimizer, scheduler, train_dl)
        mae, rmse = evaluate_rating(model, test_dl)
        dt = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]
        mk = ""
        if mae < best_mae:
            best_mae = mae; best_state = copy.deepcopy(raw_model.state_dict()); pat = 0; mk = " *"
        else: pat += 1
        print(f"  Ep {epoch+1:2d} | loss={loss:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f} | lr={lr_now:.1e} | {dt:.0f}s{mk}")
        if pat >= PATIENCE: print(f"  Early stop"); break

    raw_model.load_state_dict(best_state)
    print(f"  Best MAE: {best_mae:.4f}")
    final_mae, final_rmse = evaluate_rating(model, test_dl)
    rank = evaluate_ranking(model, test_inters, feats, item_title_embeds, item_numeric, item_raw_avg)
    results = {"mae": final_mae, "rmse": final_rmse, **rank}
    for k, v in results.items(): print(f"     {k}: {v:.4f}")
    del model, raw_model, optimizer, scheduler, train_dl, test_dl
    torch.cuda.empty_cache()
    return results

## 11. Run All Folds

In [ ]:
print("=" * 60)
print(f"  BEST-Rec v2.3 (DropoutNet p={USER_DROPOUT_P}) — GroupKFold")
print("=" * 60)

user_ids_arr = np.array([i["user_id"] for i in interactions])
indices = np.arange(len(interactions))
gkf = GroupKFold(n_splits=NUM_FOLDS)

v23_results = []
for fold_idx, (tr, te) in enumerate(gkf.split(indices, groups=user_ids_arr)):
    v23_results.append(run_fold_v23(f"v23_warm_{fold_idx}",
                                     [interactions[i] for i in tr],
                                     [interactions[i] for i in te]))

print("\n" + "=" * 60)
print("v2.3 RESULTS")
print("=" * 60)
for m in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[m] for r in v23_results]
    print(f"  {m:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

## 12. Cold-Start + All-Version Comparison

In [ ]:
def split_cold(inters, key, thr):
    counts = defaultdict(int)
    for i in inters: counts[i[key]] += 1
    cold = {k for k, c in counts.items() if c <= thr}
    return [i for i in inters if i[key] not in cold], [i for i in inters if i[key] in cold]

cu_tr, cu_te = split_cold(interactions, "user_id", COLD_USER_THRESHOLD)
print(f"Cold users: {len(cu_te):,} test interactions")
cu_res = run_fold_v23("v23_cold_user", cu_tr, cu_te) if len(cu_te) >= 10 else None

ci_tr, ci_te = split_cold(interactions, "item_id", COLD_ITEM_THRESHOLD)
print(f"Cold items: {len(ci_te):,} test interactions")
ci_res = run_fold_v23("v23_cold_item", ci_tr, ci_te) if len(ci_te) >= 10 else None

# Compare all versions
print("\n" + "=" * 70)
print(f"  ALL-VERSION COMPARISON: {DATASET.upper()}")
print("=" * 70)
for vn, vf in [("v2", "all_results.json"), ("v2.1", "v21_results.json"), ("v2.2", "v22_results.json")]:
    vp = os.path.join(CACHE_DIR, vf)
    if os.path.exists(vp):
        with open(vp) as f: vd = json.load(f)
        if "warm" in vd and vd["warm"]:
            vm = [r["mae"] for r in vd["warm"]]
            print(f"  {vn:>6s}:  MAE = {np.mean(vm):.4f} +/- {np.std(vm):.4f}")

vm23 = [r["mae"] for r in v23_results]
print(f"  {'v2.3':>6s}:  MAE = {np.mean(vm23):.4f} +/- {np.std(vm23):.4f}")

if cu_res: print(f"\n  Cold-User: MAE={cu_res['mae']:.4f}  RMSE={cu_res['rmse']:.4f}")
if ci_res: print(f"  Cold-Item: MAE={ci_res['mae']:.4f}  RMSE={ci_res['rmse']:.4f}")

# Save
def jsonify(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, dict): return {k: jsonify(v) for k, v in o.items()}
    if isinstance(o, list): return [jsonify(v) for v in o]
    return o
with open(os.path.join(CACHE_DIR, "v23_results.json"), "w") as f:
    json.dump(jsonify({"dataset": DATASET, "warm": v23_results,
                        "cold_user": cu_res, "cold_item": ci_res}), f, indent=2)
print(f"\nSaved to {CACHE_DIR}/v23_results.json")